# Biohub Cell Tracking - SUBMISSION notebook (offline, no-internet safe)

Model-agnostic submission: runs a trained **3D U-Net detector** (heatmap regression) + **rule-based
two-pass motion-aware linking + gap-closing + short-track filtering** over the test set, writes
`/kaggle/working/submission.csv`. Point it at any weights version by attaching that weights dataset --
the notebook is NOT tied to a specific model version.

**Linking upgrade (v2 fusion).** Detection is unchanged (full-res UNet heatmap -> `peak_local_max`), but
linking now uses the public rule-based stack that validated locally at **micro edge-Jaccard 0.859** (vs
0.808 for plain NN linking, same detector, same 5 val samples):
- `link_twopass` — two-pass Hungarian (tight 6um then loose 8um) with constant-velocity prediction,
- `close_gaps` — interpolate a node to bridge single-frame detection gaps,
- `prune_isolated` + `filter_short_tracks(4)` — drop unlinked / short-fragment noise.

**Before submitting:**
1. Add Input: the competition data, the `zarr3-offline` wheel dataset, and your **weights dataset**
   (a `.pt` such as `v1_UNet_best.pt`). The model architecture below MUST match the one that produced
   the weights (base channels / strides).
2. Turn **Internet OFF** and Run All to VERIFY the offline zarr install + weight load work end-to-end.
3. Save Version, then Submit to Competition.

**Density-penalty knob:** `HM_THR` controls detection count. Lower = more detections = higher base
Jaccard locally, but the leaderboard's *adjusted* score penalises over-prediction. Calibration so far
(v1: local 0.808 -> LB 0.768) suggests the penalty is mild; raise `HM_THR` only if the LB is dragged
well below local. `FILTER_MIN_LEN` and `CLOSE_GAPS` are the other main knobs.

In [ ]:
# Offline zarr>=3 install: auto-find the attached wheel dataset (no internet needed).
import glob, os, subprocess, sys
def find_wheeldir():
    for whl in glob.glob('/kaggle/input/**/zarr*.whl', recursive=True):
        return os.path.dirname(whl)
    return None
try:
    import zarr
    assert int(zarr.__version__.split('.')[0]) >= 3
    print('zarr already present:', zarr.__version__)
except Exception:
    wd = find_wheeldir()
    assert wd, 'No zarr wheel found under /kaggle/input -- attach the zarr3-offline dataset!'
    print('installing zarr offline from', wd)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '--no-index', '--find-links', wd, 'zarr'])
    import zarr
    print('installed zarr:', zarr.__version__)

In [ ]:
import numpy as np, pandas as pd, time
from dataclasses import dataclass
import torch, torch.nn as nn, torch.nn.functional as F
from scipy.optimize import linear_sum_assignment
from skimage.feature import peak_local_max

# ---------------- CONFIG ----------------
VOXEL = np.array([1.625, 0.40625, 0.40625])   # (z,y,x) um per voxel (anisotropic)
SCALE = VOXEL                                  # alias used by the rule-based linkers

# detection (match the values the model was evaluated at -> local 0.859)
NORM_PCT     = (50.0, 99.8)   # percentile normalisation, must match training
HM_THR       = 0.3            # heatmap peak threshold -> density knob (raise if LB penalised)
MIN_DISTANCE = 3              # peak_local_max separation (voxels)
MAX_PEAKS    = 40000          # per-frame cap (over-detection is cheap; won't usually be hit)
REFINE       = True           # intensity-weighted centre-of-mass sub-voxel refine
REFINE_WIN   = (1, 3, 3)      # full-res refine window

# linking + post-processing (v2 rule-based stack; validated local micro edge-J 0.859)
TIGHT_UM       = 6.0          # link_twopass tight gate
LOOSE_UM       = 8.0          # link_twopass loose gate
VEL_BLEND      = 0.5          # constant-velocity prediction blend
CLOSE_GAPS     = True         # bridge single-frame detection gaps
MAX_GAP        = 1
GAP_DIST_UM    = 6.0
FILTER_MIN_LEN = 4            # drop connected components shorter than this (nodes)
PRUNE_ISOLATED = True

# model architecture -- MUST match the training notebook that produced the weights
BASE    = 16
STRIDES = ((1, 2, 2), (1, 2, 2), (2, 2, 2), (2, 2, 2))   # downsample xy first, z later

# weights: None = auto-find under /kaggle/input (v1_UNet_best.pt / unet_heatmap.pt / unet_latest.pt / *.pt)
WEIGHTS_PATH = None

INPUT    = '/kaggle/input/competitions/biohub-cell-tracking-during-development'
TEST_DIR = os.path.join(INPUT, 'test')
OUT      = '/kaggle/working/submission.csv'
device   = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', device, '| torch', torch.__version__)

In [ ]:
# ---------------- IO + model + detection ----------------
def open_image(zarr_path):
    """Return the (T,Z,Y,X) array of an OME-Zarr sample (largest array)."""
    g = zarr.open(zarr_path, mode='r')
    if hasattr(g, 'shape'):
        return g
    best, bestsize = None, -1
    def walk(node):
        nonlocal best, bestsize
        try:
            for k in node.keys():
                sub = node[k]
                if hasattr(sub, 'shape'):
                    s = int(np.prod(sub.shape))
                    if s > bestsize: best, bestsize = sub, s
                else: walk(sub)
        except Exception: pass
    walk(g)
    return best

class ConvBlock(nn.Module):
    def __init__(self, cin, cout):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv3d(cin, cout, 3, padding=1, bias=False), nn.InstanceNorm3d(cout, affine=True), nn.LeakyReLU(0.01, True),
            nn.Conv3d(cout, cout, 3, padding=1, bias=False), nn.InstanceNorm3d(cout, affine=True), nn.LeakyReLU(0.01, True))
    def forward(self, x): return self.net(x)

class UNet3D(nn.Module):
    def __init__(self, in_ch=1, base=BASE, strides=STRIDES):
        super().__init__()
        n = len(strides); chs = [base*(2**i) for i in range(n+1)]
        self.enc, self.downs = nn.ModuleList(), nn.ModuleList()
        cin = in_ch
        for i in range(n):
            self.enc.append(ConvBlock(cin, chs[i]))
            self.downs.append(nn.Conv3d(chs[i], chs[i], kernel_size=strides[i], stride=strides[i]))
            cin = chs[i]
        self.bottleneck = ConvBlock(chs[n-1], chs[n])
        self.ups, self.dec = nn.ModuleList(), nn.ModuleList()
        for i in reversed(range(n)):
            self.ups.append(nn.ConvTranspose3d(chs[i+1], chs[i], kernel_size=strides[i], stride=strides[i]))
            self.dec.append(ConvBlock(chs[i]*2, chs[i]))
        self.head = nn.Conv3d(chs[0], 1, 1)
    def forward(self, x):
        skips = []
        for enc, down in zip(self.enc, self.downs):
            x = enc(x); skips.append(x); x = down(x)
        x = self.bottleneck(x)
        for up, dec, skip in zip(self.ups, self.dec, reversed(skips)):
            x = up(x)
            if x.shape[2:] != skip.shape[2:]:
                x = F.interpolate(x, size=skip.shape[2:], mode='trilinear', align_corners=False)
            x = dec(torch.cat([x, skip], dim=1))
        return self.head(x)   # raw logits; sigmoid at inference

def _normalize(vol, norm_pct=NORM_PCT):
    v = vol.astype(np.float32)
    lo, hi = np.percentile(v, norm_pct)
    return np.clip((v - lo) / (hi - lo + 1e-6), 0, 1).astype(np.float32)   # keep float32 (AMP-safe)

@torch.no_grad()
def predict_heatmap(model, vol, norm_pct=NORM_PCT):
    v = _normalize(vol, norm_pct)
    x = torch.from_numpy(np.ascontiguousarray(v)[None, None]).float().to(device)
    with torch.amp.autocast(device, enabled=(device == 'cuda')):
        y = torch.sigmoid(model(x))
    return y[0, 0].float().cpu().numpy()

def detect_frames(model, arr, thr=HM_THR, min_distance=MIN_DISTANCE, max_peaks=MAX_PEAKS,
                  refine=REFINE, win=REFINE_WIN):
    """Per-frame detection -> list over t of (N,3) voxel coords (z,y,x)."""
    model.eval(); frames = []
    for t in range(arr.shape[0]):
        vol = np.asarray(arr[t]).astype(np.float32)
        hm = predict_heatmap(model, vol)
        pk = peak_local_max(hm, min_distance=min_distance, threshold_abs=thr,
                            exclude_border=False, num_peaks=max_peaks)
        coords = pk.astype(np.float64)
        if refine and len(coords):
            coords = refine_centroids(vol, coords, win=win)
        frames.append(coords)
    return frames

# ---------------- Linking + post-processing (rule-based v2 stack) ----------------
@dataclass
class TrackGraph:
    node_t: np.ndarray; node_z: np.ndarray; node_y: np.ndarray; node_x: np.ndarray
    node_ids: np.ndarray; edges: np.ndarray; meta: dict
    @property
    def n_nodes(self): return len(self.node_ids)
    @property
    def n_edges(self): return len(self.edges)

def refine_centroids(vol, coords, win=(1, 3, 3)):
    """Intensity-weighted local centre-of-mass refinement on the original-resolution volume."""
    if len(coords) == 0:
        return coords
    Z, Y, X = vol.shape
    out = coords.copy().astype(np.float64)
    wz, wy, wx = win
    for i, (z, y, x) in enumerate(coords):
        z, y, x = int(round(z)), int(round(y)), int(round(x))
        z0, z1 = max(0, z - wz), min(Z, z + wz + 1)
        y0, y1 = max(0, y - wy), min(Y, y + wy + 1)
        x0, x1 = max(0, x - wx), min(X, x + wx + 1)
        patch = vol[z0:z1, y0:y1, x0:x1].astype(np.float64)
        s = patch.sum()
        if s <= 0: continue
        zz = np.arange(z0, z1)[:, None, None]
        yy = np.arange(y0, y1)[None, :, None]
        xx = np.arange(x0, x1)[None, None, :]
        out[i, 0] = (patch * zz).sum() / s
        out[i, 1] = (patch * yy).sum() / s
        out[i, 2] = (patch * xx).sum() / s
    return out

def link_twopass(frames, tight_um=6.0, loose_um=8.0, vel_blend=0.5):
    node_ids = []; node_t = []; node_z = []; node_y = []; node_x = []; frame_ids = []; nid = 1
    for t, coords in enumerate(frames):
        ids = []
        for (z, y, x) in coords:
            node_ids.append(nid); node_t.append(t); node_z.append(z); node_y.append(y); node_x.append(x)
            ids.append(nid); nid += 1
        frame_ids.append(ids)
    def _hun(P, C, pred, pi, ci, gate):
        if len(pi) == 0 or len(ci) == 0: return []
        BIG = 1e9
        Draw = np.sqrt(((P[pi][:, None] - C[ci][None]) ** 2).sum(2))
        D = np.sqrt(((pred[pi][:, None] - C[ci][None]) ** 2).sum(2))
        cost = np.where(Draw > gate, BIG, D)
        ri, rc = linear_sum_assignment(cost)
        return [(int(pi[r]), int(ci[c])) for r, c in zip(ri, rc) if cost[r, c] < BIG]
    edges = []; prev_xyz = np.zeros((0, 3)); prev_ids = []; prev_vel = None
    for t in range(len(frames)):
        curr = np.asarray(frames[t], float).reshape(-1, 3); curr_ids = frame_ids[t]
        if t > 0 and len(prev_ids) and len(curr):
            P = prev_xyz * SCALE[None, :]; C = curr * SCALE[None, :]
            pred = P + (vel_blend * prev_vel if prev_vel is not None else 0.0)
            N, M = len(P), len(C)
            links = _hun(P, C, pred, np.arange(N), np.arange(M), min(tight_um, loose_um))
            up = {p for p, _ in links}; uc = {c for _, c in links}
            fp = np.array([i for i in range(N) if i not in up], int)
            fc = np.array([j for j in range(M) if j not in uc], int)
            links += _hun(P, C, pred, fp, fc, loose_um)
            vel = np.zeros((N, 3)); nv = np.zeros((M, 3))
            for p, c in links:
                edges.append((prev_ids[p], curr_ids[c])); vel[p] = (curr[c] - prev_xyz[p]) * SCALE
            for p, c in links:
                nv[c] = vel[p]
            prev_vel = nv
        else:
            prev_vel = None
        prev_xyz, prev_ids = curr, curr_ids
    return TrackGraph(node_t=np.array(node_t, np.int64), node_z=np.array(node_z, float),
                      node_y=np.array(node_y, float), node_x=np.array(node_x, float),
                      node_ids=np.array(node_ids, np.int64),
                      edges=np.array(edges, np.int64).reshape(-1, 2), meta={})

def close_gaps(frames, g, max_gap=1, gap_dist_um=8.0):
    """Insert an interpolated node to bridge single-frame detection gaps (all GT edges are dt=1)."""
    if g.n_edges == 0:
        return g
    coords = {int(nid): (int(g.node_t[i]), g.node_z[i], g.node_y[i], g.node_x[i])
              for i, nid in enumerate(g.node_ids)}
    has_out = set(int(s) for s, _ in g.edges)
    has_in = set(int(t) for _, t in g.edges)
    ends_by_t = {}; starts_by_t = {}
    for nid, (t, z, y, x) in coords.items():
        if nid not in has_out: ends_by_t.setdefault(t, []).append(nid)
        if nid not in has_in: starts_by_t.setdefault(t, []).append(nid)
    new_nodes = []; new_edges = []
    next_id = int(g.node_ids.max()) + 1 if g.n_nodes else 1
    for gap in range(1, max_gap + 1):
        for t, ends in ends_by_t.items():
            starts = starts_by_t.get(t + gap + 1)
            if not starts: continue
            ec = np.array([[coords[e][1], coords[e][2], coords[e][3]] for e in ends]) * SCALE
            sc = np.array([[coords[s][1], coords[s][2], coords[s][3]] for s in starts]) * SCALE
            d = np.sqrt(((ec[:, None, :] - sc[None, :, :]) ** 2).sum(axis=2))
            thr = gap_dist_um * (gap + 1)
            big = thr * 1000 + 1
            cost = np.where(d <= thr, d, big)
            ri, ci = linear_sum_assignment(cost)
            used_s = set()
            for r, c in zip(ri, ci):
                if d[r, c] > thr or ends[r] in has_out or starts[c] in used_s: continue
                e_id, s_id = ends[r], starts[c]
                te, ze, ye, xe = coords[e_id]; ts, zs, ys, xs = coords[s_id]
                prev = e_id
                for k in range(1, gap + 1):
                    frac = k / (gap + 1)
                    zi = ze + (zs - ze) * frac; yi = ye + (ys - ye) * frac; xi = xe + (xs - xe) * frac
                    nid = next_id; next_id += 1
                    new_nodes.append((te + k, zi, yi, xi, nid)); new_edges.append((prev, nid)); prev = nid
                new_edges.append((prev, s_id)); has_out.add(e_id); used_s.add(c)
    if not new_nodes:
        return g
    nt = np.concatenate([g.node_t, np.array([n[0] for n in new_nodes], dtype=np.int64)])
    nz = np.concatenate([g.node_z, np.array([n[1] for n in new_nodes])])
    ny = np.concatenate([g.node_y, np.array([n[2] for n in new_nodes])])
    nx = np.concatenate([g.node_x, np.array([n[3] for n in new_nodes])])
    nid = np.concatenate([g.node_ids, np.array([n[4] for n in new_nodes], dtype=np.int64)])
    edges = np.concatenate([g.edges, np.array(new_edges, dtype=np.int64).reshape(-1, 2)])
    return TrackGraph(node_t=nt, node_z=nz, node_y=ny, node_x=nx, node_ids=nid, edges=edges, meta=g.meta)

def prune_isolated(g):
    if g.n_edges == 0: return g
    used = set(int(x) for x in g.edges.reshape(-1))
    keep = np.array([i for i, nid in enumerate(g.node_ids) if int(nid) in used])
    if len(keep) == len(g.node_ids): return g
    return TrackGraph(node_t=g.node_t[keep], node_z=g.node_z[keep], node_y=g.node_y[keep],
                      node_x=g.node_x[keep], node_ids=g.node_ids[keep], edges=g.edges, meta=g.meta)

def filter_short_tracks(g, min_len):
    if g.n_edges == 0 or min_len <= 1: return g
    parent = {int(n): int(n) for n in g.node_ids}
    def find(a):
        while parent[a] != a:
            parent[a] = parent[parent[a]]; a = parent[a]
        return a
    for s, t in g.edges.reshape(-1, 2):
        ra, rb = find(int(s)), find(int(t))
        if ra != rb: parent[ra] = rb
    comp = {}
    for n in g.node_ids:
        comp.setdefault(find(int(n)), []).append(int(n))
    keep = set()
    for members in comp.values():
        if len(members) >= min_len: keep.update(members)
    idx = [i for i, n in enumerate(g.node_ids) if int(n) in keep]
    keepset = set(int(g.node_ids[i]) for i in idx)
    edges = np.array([(int(s), int(t)) for s, t in g.edges.reshape(-1, 2)
                      if int(s) in keepset and int(t) in keepset], dtype=np.int64).reshape(-1, 2)
    return TrackGraph(node_t=g.node_t[idx], node_z=g.node_z[idx], node_y=g.node_y[idx],
                      node_x=g.node_x[idx], node_ids=g.node_ids[idx], edges=edges, meta=g.meta)

def track_movie(model, arr):
    """Full per-movie pipeline: detect -> two-pass link -> close gaps -> prune -> filter short."""
    frames = detect_frames(model, arr)
    g = link_twopass(frames, tight_um=TIGHT_UM, loose_um=LOOSE_UM, vel_blend=VEL_BLEND)
    if CLOSE_GAPS:     g = close_gaps(frames, g, max_gap=MAX_GAP, gap_dist_um=GAP_DIST_UM)
    if PRUNE_ISOLATED: g = prune_isolated(g)
    if FILTER_MIN_LEN > 1: g = filter_short_tracks(g, FILTER_MIN_LEN)
    return g

In [ ]:
# ---------------- Load weights (auto-find; handles state_dict OR full checkpoint) ----------------
def find_weights():
    if WEIGHTS_PATH:
        return WEIGHTS_PATH
    for pat in ('v1_UNet_best.pt', 'unet_heatmap.pt', 'unet_latest.pt', '*.pt'):
        hits = sorted(glob.glob(f'/kaggle/input/**/{pat}', recursive=True))
        if hits:
            return hits[0]
    return None

def extract_state(ck):
    if isinstance(ck, dict) and ('best_model' in ck or 'model' in ck):
        return ck.get('best_model') or ck['model']
    return ck

wpath = find_weights()
assert wpath, 'No .pt weights found under /kaggle/input -- attach your weights dataset (e.g. v1_UNet_best.pt).'
model = UNet3D().to(device)
model.load_state_dict(extract_state(torch.load(wpath, map_location=device)))
model.eval()
print('loaded weights:', wpath)

In [ ]:
# ---------------- Generate submission for ALL test datasets ----------------
test_zarrs = sorted(glob.glob(os.path.join(TEST_DIR, '*.zarr')))
print('test datasets:', len(test_zarrs))

rows = []; gid = 0; t0 = time.time()
for zp in test_zarrs:
    ds = os.path.basename(zp)[:-5]
    arr = open_image(zp)
    g = track_movie(model, arr)
    for i in range(g.n_nodes):
        rows.append((gid, ds, 'node', int(g.node_ids[i]), int(g.node_t[i]),
                     int(round(g.node_z[i])), int(round(g.node_y[i])), int(round(g.node_x[i])), -1, -1)); gid += 1
    for (s, tg) in g.edges:
        rows.append((gid, ds, 'edge', -1, -1, -1, -1, -1, int(s), int(tg))); gid += 1
    print(f'  {ds}: {g.n_nodes} nodes, {g.n_edges} edges  ({time.time()-t0:.0f}s)')

cols = ['id','dataset','row_type','node_id','t','z','y','x','source_id','target_id']
sub = pd.DataFrame(rows, columns=cols)
sub.to_csv(OUT, index=False)
print('\nwrote', OUT, sub.shape)
print(sub.head(6).to_string(index=False))